# 🎬 Bitrate Backtrack — Cloud Video Assembly Engine
### Automated Countdown Compilation with Free Nvidia T4 GPU (Google Colab)

This notebook completely solves the **slow home internet upload** and **VPS CPU freeze** problems:
1. **Free Nvidia T4 GPU (16GB VRAM):** Video rendering uses hardware NVENC acceleration (`h264_nvenc`) — renders a 30-minute 1080p60 countdown in **~3-5 minutes** without any CPU freezing.
2. **Google Datacenter Speed (800+ Mbps):** Downloads raw YouTube gameplay clips in seconds.
3. **Zero Local Bandwidth:** You only upload tiny voiceover MP3 files (5-10 MB total). The finished 2-4 GB master video is saved directly to your **Google Drive** internally.

## Step 1: Verify Free Nvidia GPU & Mount Google Drive
Make sure your Colab runtime has a GPU: Go to **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Check GPU status
!nvidia-smi

# Mount your Google Drive to save the final video and load voiceovers
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone Repo & Install Requirements

In [ ]:
# Clone repository
!git clone https://github.com/Clarionxk/segment-dw.git /content/segment-dw
%cd /content/segment-dw

# Install dependencies and ensure FFmpeg is up to date
!pip install -r requirements.txt
!apt-get update -qq && apt-get install -y ffmpeg -qq

## Step 3: Load Voiceover Audio Files
You can either:
- Place your VO files (`01.mp3`, `02.mp3`, etc.) in a folder named `voiceovers` in your Google Drive (`Google Drive/voiceovers`), OR
- Upload them using the file manager on the left panel into `/content/segment-dw/voiceovers/`.

In [ ]:
import os, shutil
os.makedirs('voiceovers', exist_ok=True)

# If you have a folder named 'voiceovers' in your Google Drive root:
drive_vo_path = '/content/drive/MyDrive/voiceovers'
if os.path.exists(drive_vo_path):
    for f in os.listdir(drive_vo_path):
        src = os.path.join(drive_vo_path, f)
        if os.path.isfile(src):
            shutil.copy(src, os.path.join('voiceovers', f))
    print(f"[+] Copied {len(os.listdir('voiceovers'))} VO files from Google Drive.")
else:
    print("[*] Ready. Place your VO files into /content/segment-dw/voiceovers/")

!ls -la voiceovers

## Step 4: Download Gameplay Clips at Datacenter Speed (800+ Mbps)
Edit `videoplan1.md` if needed, then run the downloader.

In [ ]:
# Download all clips defined in videoplan1.md
!python download.py videoplan1.md --res 1080p

## Step 5: Assemble Video & Duck Audio with GPU Hardware Acceleration
This automatically:
1. Pairs video clips with voiceovers.
2. Ducks gameplay audio to 18% during narration, then swells to 100% for raw gameplay sound effects.
3. Uses the free Nvidia T4 GPU (`--gpu`) for lightning-fast encoding.
4. Concatenates all countdown items into `output/master_compilation.mp4`.
5. Generates YouTube chapter timestamps.

In [ ]:
!python assemble.py videoplan1.md --gpu --solo-sfx 10

## Step 6: Save Master Video Directly to Google Drive
Transfers internally inside Google's network in seconds — zero home internet bandwidth used!

In [ ]:
# Copy final master video and chapter markers directly to Google Drive
!cp output/master_compilation.mp4 /content/drive/MyDrive/master_compilation.mp4
!cp output/youtube_chapters.txt /content/drive/MyDrive/youtube_chapters.txt

print("\n🎉 SUCCESS! Your finished compilation video is saved in your Google Drive: MyDrive/master_compilation.mp4")
print("Chapter markers saved to: MyDrive/youtube_chapters.txt")